# P2｜BEATs Native Encoder

**状态：production adapter Code READY；tracked manifest 已冻结，但 canonical `.cache/multidataset_pipeline/assets/P2/...` 尚待 provision，Asset/L40 CUDA/实验审批 HOLD；首轮 core。** 目的：与 P1 完全匹配，只把四数据集共享的 frozen candidate encoder package 从 AST 换为 BEATs。旧 `result/pafa...` 路径不是 fallback。

## 唯一变量、匹配对照与四数据集边界

matched comparator=P1。四数据集都使用相同 2.0 s / 1.0 s source-time windows、同一个 frozen BEATs、同一个 trainable Linear 768→256 projector 与 dataset-native heads。除 AST+frontend package→BEATs+frontend package 外，window、split/group、head、sampler、scope、budget、seed、selection 与 metrics 必须完全匹配。

ICBHI/SPRSound/KAUH 对 valid window embeddings masked mean；HF 保留 projected sequence [B,K,256] 并输出 I/E/CAS/DAS [B,K,4]。这是四数据集 shared encoder/projector，但不是统一标签任务。

- ICBHI cycle flat4 [B,4]；SPRSound event binary [B,2] 与 raw7 [B,7]；KAUH recording raw9 [B,9]，B/D/E 同 patient group。
- HF window-center target 使用 paper-native one-vs-rest rasterization；constructed negative 不等于 raw normal，missing/unknown 继续 fail closed。
- 输入/输出：16 kHz native units → windows [B,K,32000] → BEATs embeddings [B,K,768] → shared projected [B,K,256] → native aggregation/heads。P2−P1 只能解释为 encoder+frontend package replacement。

In [ ]:
from pathlib import Path
import os
from baseline.multidataset_pipeline.preflight import P1_P5_SELECTION_RULE, P1_P5_UPDATE_BUDGET, freeze_receipt
from baseline.multidataset_pipeline.adapter_factory import AdapterFactoryConfig, build_production_adapter
from baseline.multidataset_pipeline.real_subtrain_provider import build_frozen_provider_index, build_real_subtrain_preflight_batches
from baseline.multidataset_pipeline.train_shared_window import TrainingRunnerConfig
from baseline.multidataset_pipeline.asset_manifest import load_adapter_asset_manifest
from baseline.multidataset_pipeline.embedding_cache import FrozenEmbeddingCache, EmbeddingCacheIdentity
from baseline.multidataset_pipeline.runner_embedding_cache import build_or_load_runner_embedding_caches
from baseline.multidataset_pipeline.terminal_scoring import ProductionTerminalScorer, NATIVE_TASKS

PIPELINE = {
    "id": "P2",
    "comparator": "P1",
    "only_change": "four_dataset_shared_encoder_package: AST_to_BEATs",
    "seed": 20260728,
    "split_policy": "reuse_P1_immutable_receipts",
    "provider_schema": "real_frozen_provider_identity_v2",
    "runner_schema": "shared_window_training_v5",
    "encoder_scope": "frozen_pretrained_BEATs",
    "shared_projector": "minimal_linear_projector_768_to_256_match_P1",
    "shared_projector_bias": True,
    "shared_projector_lanes": ["ICBHI", "SPRSound", "HF", "KAUH"],
    "window_policy": "source_time_2s_window_1s_stride_match_P1",
    "sampler": "four_dataset_source_proportional_match_P1",
    "hf_lane": "shared_projected_window_sequence_to_temporal4_head",
    "hf_uses_shared_projector": True,
    "trainable_scope": "one_shared_projector_768_to_256_plus_four_dataset_native_heads_match_P1",
    "contract_modules": ["baseline.multidataset_pipeline.real_subtrain_provider", "baseline.multidataset_pipeline.sliding_window", "baseline.multidataset_pipeline.window_encoder", "baseline.multidataset_pipeline.beats_temporal", "baseline.multidataset_pipeline.beats_window_encoder", "baseline.multidataset_pipeline.adapter_factory", "baseline.multidataset_pipeline.asset_manifest", "baseline.multidataset_pipeline.embedding_cache", "baseline.multidataset_pipeline.runner_embedding_cache", "baseline.multidataset_pipeline.terminal_scoring", "baseline.multidataset_pipeline.train_shared_window", "baseline.multidataset_pipeline.l40_preflight"],
    "engineering_tests": ["tests/test_multidataset_pipeline.py::ProductionWindowAdapterTest", "tests/test_shared_window_execution.py", "tests/test_terminal_scoring_and_cache.py"],
    "asset_manifest": "baseline/multidataset_pipeline/adapter_assets.json",
    "canonical_source": ".cache/multidataset_pipeline/assets/P2/source/repo",
    "canonical_checkpoint": ".cache/multidataset_pipeline/assets/P2/checkpoints/BEATs_iter3_plus_AS2M.pt",
    "embedding_cache": "subtrain_validation_only_frozen_eval_deterministic",
    "terminal_native_tasks": list(NATIVE_TASKS),
    "batch_size": 8,
    "update_budget": P1_P5_UPDATE_BUDGET,
    "validation_interval_updates": 1725,
    "selection": P1_P5_SELECTION_RULE,
    "output_dir": "result/reproduce/P2_shared_window_seed20260728",
    "receipt_path": "result/reproduce/P2_shared_window_seed20260728/<phase>/<approval_receipt_sha256>/run_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P2_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

P1–P5 共同冻结 seed=20260728、batch size=8、86250 updates、每 1725 updates 做 validation-only selection；不得报告跨数据集 pooled Score。P1/P2 共用真实 frozen provider、Adam(lr=5e-5, weight_decay=1e-6) Proposed Benchmark Policy、resume/checkpoint/logging与exact terminal scorer gate。科学 config SHA 不因 phase 改变，但 smoke/full 的全部执行 artifacts 分别写入 `<output_dir>/<phase>/<approval_receipt_sha256>/`；fresh root 禁止复用，full resume 只能续接同一 execution identity 与最新 checkpoint/log receipt 链。P2 full 在 update 1 前强制闭合 subtrain+validation×四 lane 8项 frozen embedding cache，之后训练/validation只走共享 `.cache` identity root；smoke 明确 uncached。BEATs checkpoint/source revision/SHA、2 s exact valid-patch mask、B×K restore、lineage 与单窗口真实 CPU smoke 是旧路径下的工程证据；正式运行必须先把完全相同的 bytes/source revision provision 到 tracked manifest 的 canonical `.cache` 路径并重新 audit，禁止 silent fallback。当前 production terminal provider registration 缺失，机器可读 terminal HOLD。随后仍须 L40 CUDA ≤2 subtrain batches/0 update preflight、完整训练批准和独立 verifier。配置 verifier 要证明除四数据集 shared encoder package 外零差异。

In [ ]:
PREFLIGHT = freeze_receipt()
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P2 fail closed: approval receipt is missing")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "approval": str(approval_path), "execute": False}


## 输出、receipt 与结果表

receipt 增加 P1 parity hash、tracked asset-manifest identity、canonical BEATs checkpoint/frontend/window-adapter hashes、embedding-cache artifact receipts、exact selection/checkpoint/terminal binding、per-dataset unit/window/valid counts、HF target semantics、五个 native task support/denominator 与 independent verifier。结果必须按任务报告，不能 pooled。

| Comparison | Native task | Result | Decision |
|---|---|---:|---|
| P2−P1 | ICBHI / SPRSound / HF / KAUH native tasks | Not run | HOLD |

**Test Result = Not run。Decision = HOLD。Claim boundary：只能解释四数据集 shared-window encoder+frontend package 的 AST→BEATs 替换。**